In [1]:
import ase
from ase.build import graphene_nanoribbon
from ase.cluster import Icosahedron
import py3Dmol
import io
import psi4
print("Building components...")

Building components...


In [2]:
# Create the Carbon Flake
flake = graphene_nanoribbon(4,4, type ='armchair',saturated=True,vacuum=5.0)
#Create the Iron Catalyst Cluster(Fe13)
fe13 = Icosahedron('Fe', noshells=2, latticeconstant=2.52)


In [3]:
#Create the FeCl3 Vapor Molecule
fecl3 =ase.Atoms('FeCl3',positions=[[0.0, 0.0, 0.0],       # Fe
                             [2.14, 0.0, 0.0],      # Cl 1
                             [-1.07, 1.853, 0.0],   # Cl 2
                             [-1.07, -1.853, 0.0]])#Cl3

In [4]:
flake.center(about=(0., 0., 0.))
fe13.center(about=(0., 0., 0.))
fecl3.center(about=(0., 0., 0.))

flake.rotate(90,'x',center='COM')

flake.positions -= flake.positions.mean(axis=0)
fe13.positions -= fe13.positions.mean(axis=0)
fecl3.positions -= fecl3.positions.mean(axis=0)

fe13.positions[:, 2] += 4.5
fecl3.positions[:, 2] += 9.6
combined_system = flake + fe13 + fecl3

In [5]:
# 6. Visualize with py3Dmol
xyz_file = io.StringIO()
ase.io.write(xyz_file, combined_system, format='xyz')
xyz_data = xyz_file.getvalue()
viewer = py3Dmol.view(width=800, height=500)
viewer.addModel(xyz_data, 'xyz')
viewer.setStyle({'sphere': {'scale': 0.25}, 'stick': {'radius': 0.1}})
viewer.zoomTo()
viewer.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [6]:
import numpy as np
SAFE_DISTANCE = 2.0
def calculate_minimum_clearance(comp1,comp2,name1,name2):
    min_dist = float('inf')
    for pos1 in comp1.positions:
        for pos2 in comp2.positions:
            dist = np.linalg.norm(pos1 - pos2)
            if dist < min_dist:
                min_dist = dist
    status = "Pass" if min_dist >= SAFE_DISTANCE else "Fail"
    print(f"Clearance [{name1}] to [{name2}]: {min_dist:.2f} Å ----> {status}")
    return min_dist
print("--- Steric Clearance Checks ---")

dist_flake_fe = calculate_minimum_clearance(flake, fe13, "Carbon Flake", "Fe13 Cluster")
dist_fe_gas = calculate_minimum_clearance(fe13, fecl3, "Fe13 Cluster", "FeCl3 Vapor")
print("\n Electronic State Summary")
total_atoms = len(combined_system)
valence_electrons = 0
for atom in combined_system:
    if atom.symbol == 'Fe': valence_electrons+= 8
    elif atom.symbol == 'C': valence_electrons += 4
    elif atom.symbol == 'H': valence_electrons +=1 
    elif atom.symbol == 'Cl': valence_electrons +=7

print(f"Total Valence Electrons: {valence_electrons}")

--- Steric Clearance Checks ---
Clearance [Carbon Flake] to [Fe13 Cluster]: 3.06 Å ----> Pass
Clearance [Fe13 Cluster] to [FeCl3 Vapor]: 3.70 Å ----> Pass

 Electronic State Summary
Total Valence Electrons: 405


In [7]:
import psi4
from ase.io import write

# Wipe all internal C++ timers and memory from previous runs
psi4.core.clean()

print("Initializing Psi4 for Optimization...")

# 1. Allocate computational resources
psi4.set_memory('6 GB')
psi4.set_num_threads(4) 

# 2. Tell Psi4 to write all the quantum mechanical math to a log file
psi4.core.set_output_file('FeCl3_optimization.dat', False)

# 3. Extract the coordinates of JUST the FeCl3 molecule
# Charge = 0, Multiplicity = 6
xyz_string = "0 6\n"
for atom in fecl3:
    xyz_string += f"{atom.symbol} {atom.position[0]} {atom.position[1]} {atom.position[2]}\n"

xyz_string += "symmetry c1\n"

# 4. Load the geometry into Psi4
molecule = psi4.geometry(xyz_string)

# 5. Set the theoretical models (DFT Functional and Basis Set)
psi4.set_options({
    'basis': 'def2-svp',   
    'reference': 'uhf',    
    'scf_type': 'df'       
})

# 6. Run the Geometry Optimization
print("Starting DFT structural relaxation. This may take a few minutes...")
final_energy = psi4.optimize('b3lyp')
write('Reactant_Baseline_Optimized.xyz', combined_system)

print(f"\nOptimization Complete!")
print(f"Final Ground State Energy (E_Reactants): {final_energy} Hartrees")

Some dependencies such as QCElemental have not yet finished migration to pydantic v2. If issues are encountered please downgrade pydantic or upgrade QCElemental as appropriate


Initializing Psi4 for Optimization...
Starting DFT structural relaxation. This may take a few minutes...


	Previous geometry is closer to target in internal coordinates, so using that one.

	Best geometry has RMS(Delta(q)) = 8.42e-09



Optimizer: Optimization complete!

Optimization Complete!
Final Ground State Energy (E_Reactants): -2643.914147756004 Hartrees


In [ ]:
import psi4
psi4.core.clean()
psi4.set_memory('400MB')
psi4.core.set_num_threads(4)
psi4.core.set_output_file('FeCl3_Gas_900C.dat', False)

fecl3_string = """
0 6
Fe   0.0000   0.0000   0.0000
Cl   2.1400   0.0000   0.0000
Cl  -1.0700   1.8533   0.0000
Cl  -1.0700  -1.8533   0.0000
symmetry c1
no_reorient
no_com
"""

fecl3_gas = psi4.geometry(fecl3_string)
psi4.set_options({
    'basis': 'def2-svp',
    'reference': 'uhf',
    'scf_type': 'df', 
    'T': 1173.15
})
print("Optimizing FeCl3 Gas...")
psi4.optimize('b3lyp', molecule=fecl3_gas)
print("Calculating Frequencies and Entropy...")
psi4.frequency('b3lyp', molecule=fecl3_gas)

print("\nDone! Open FeCl3_Gas_900C.dat to see the Entropy (S).")

Optimizing FeCl3 Gas...


	Previous geometry is closer to target in internal coordinates, so using that one.

	Best geometry has RMS(Delta(q)) = 2.28e-11



Optimizer: Optimization complete!
Calculating Frequencies and Entropy...

Done! Open FeCl3_Gas_Thermo.dat to see the Entropy (S).


In [8]:
import psi4
psi4.core.clean()
psi4.set_memory('4GB')
psi4.core.set_num_threads(4)
psi4.core.set_output_file('FeCl3_Gas_1000C.dat', False)

fecl3_string = """
0 6
Fe   0.0000   0.0000   0.0000
Cl   2.1400   0.0000   0.0000
Cl  -1.0700   1.8533   0.0000
Cl  -1.0700  -1.8533   0.0000
symmetry c1
no_reorient
no_com
"""

fecl3_gas = psi4.geometry(fecl3_string)
psi4.set_options({
    'basis': 'def2-svp',
    'reference': 'uhf',
    'scf_type': 'df', 
    'T': 1273.15
})
print("Optimizing FeCl3 Gas...")
psi4.optimize('b3lyp', molecule=fecl3_gas)
print("Calculating Frequencies and Entropy...")
psi4.frequency('b3lyp', molecule=fecl3_gas)

print("\nDone! Open FeCl3_Gas_1000C.dat to see the Entropy (S).")

Optimizing FeCl3 Gas...
Optimizer: Optimization complete!
Calculating Frequencies and Entropy...

Done! Open FeCl3_Gas_1000C.dat to see the Entropy (S).


In [9]:
import psi4
psi4.core.clean()
psi4.set_memory('400MB')
psi4.core.set_num_threads(4)
psi4.core.set_output_file('FeCl3_Gas_1100C.dat', False)

fecl3_string = """
0 6
Fe   0.0000   0.0000   0.0000
Cl   2.1400   0.0000   0.0000
Cl  -1.0700   1.8533   0.0000
Cl  -1.0700  -1.8533   0.0000
symmetry c1
no_reorient
no_com
"""

fecl3_gas = psi4.geometry(fecl3_string)
psi4.set_options({
    'basis': 'def2-svp',
    'reference': 'uhf',
    'scf_type': 'df', 
    'T': 1373.15
})
print("Optimizing FeCl3 Gas...")
psi4.optimize('b3lyp', molecule=fecl3_gas)
print("Calculating Frequencies and Entropy...")
psi4.frequency('b3lyp', molecule=fecl3_gas)

print("\nDone! Open FeCl3_Gas_1100C.dat to see the Entropy (S).")

Optimizing FeCl3 Gas...
Optimizer: Optimization complete!
Calculating Frequencies and Entropy...

Done! Open FeCl3_Gas_1100C.dat to see the Entropy (S).


1100C Reaction Simulation


In [ ]:
import numpy as np
import ase 
from ase.build import graphene_nanoribbon
from ase.cluster import Icosahedron
from ase.constraints import FixAtoms
from ase.calculators.psi4 import Psi4
from ase.optimize import LBFGS
from ase.io import write
import psi4

psi4.core.clean()
print("1. Forging the baseline surface...")
flake = graphene_nanoribbon(4,4, type ='armchair',saturated=True,vacuum=5.0)
flake.rotate(90,'x',center='COM')
flake.positions -= flake.positions.mean(axis=0)

print("2. Simulating 1100 C Thermal Lattice Destruction...")
distances = np.linalg.norm(flake.positions, axis=1)
#Sort the atoms by distance and selcting the 4 central-most Carbon atoms 
central_indices = np.argsort(distances)[:4]
#Extract their exact coordinates before delete them
dissolved_carbon_coords = flake.positions[central_indices]
del flake[central_indices]
print("3. building the Carbide Cluster...")
fe_carbide = Icosahedron('Fe', noshells =1, latticeconstant=2.52)
fe_carbide.positions -= fe_carbide.positions.mean(axis=0)
#Shift the iron cluster up slightly
fe_carbide.positions[:, 2] += 4.5
#Spawn the dissolved carbon atoms into the iron cluster's coordinates space
intersititial_carbons = ase.Atoms('C4', positions=dissolved_carbon_coords)
intersititial_carbons.positions[:,2] += 4.5

#Re-introduce the vapor
fecl3 = ase.Atoms('FeCl3',
                  positions=[[0.0, 0.0, 0.0], [2.14, 0.0, 0.0], 
                             [-1.07, 1.853, 0.0], [-1.07, -1.853, 0.0]]

)
fecl3.positions -= fecl3.positions.mean(axis=0)
fecl3.positions[:, 2] += 9.6

failure_state_system = flake+fe_carbide+intersititial_carbons+fecl3
print("4. Applying Constraints and Setting Calculator...")

num_flake_atoms = len(flake)
constraint = FixAtoms(indices=range(num_flake_atoms))
failure_state_system.set_constraint(constraint)
write('Phase2_Initial_Guess.xyz', failure_state_system)
# Setup Psi4 - Same settings to ensure thermodynamic comparability
calc = Psi4(method='pbe', 
            basis='def2-svp', 
            reference='uhf', 
            multiplicity=50, 
            charge=0,
            memory='1200MB',
            threads=4,
            maxiter=300,
            soscf = True
            )

failure_state_system.calc = calc

print("Starting 1100°C Carbide Trap Optimization...")
optimizer = LBFGS(failure_state_system, trajectory='1100C_Carbide_Trap.traj')
optimizer.run(fmax=0.15) 

print("\nOptimization Complete!")
write('1100C_Carbide_Trap_Optimized.xyz', failure_state_system)
print("Failure State geometry saved!")


1. Forging the baseline surface...
2. Simulating 1100 C Thermal Lattice Destruction...
3. building the Carbide Cluster...
4. Applying Constraints and Setting Calculator...
Starting 1100°C Carbide Trap Optimization...


SCFConvergenceError: Could not converge SCF iterations in 100 iterations.